# ????? 2026? 9? ???? / ?????? ??

???? ???? ??? 5? ?? ???? ?? ????????. ????? ???? ?????????? ??? 24?? ??, ??????? 2026? 9? ? ?? ?? ??? ?????. ??? ?? ???? ?? ?? ??? ?? ???? ??? ????.

- ????: [????? OpenAPI](https://www.data.go.kr/data/15164444/openapi.do), ???? 202609
- ??????: [????? CSV](https://www.data.go.kr/data/3033254/fileData.do), ??? 2026-09-30
- ??: ????? 2025-01-01 ??? GeoJSON?? ?? 206? ? ??
- ????: V-World WMTS (?? ?? ??? `.env`?? ??, ?? HTML/?? ???? ???? ??)


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import folium
from dotenv import load_dotenv
from IPython.display import IFrame, display

# ???? ??? ?? ?? ?? ?? ???? ???? data/? ????.
cwd = Path.cwd()
if (cwd / "data" / "busan_living_population_202609.csv").exists():
    project_dir = cwd
elif (cwd / "shrinking-busan" / "data" / "busan_living_population_202609.csv").exists():
    project_dir = cwd / "shrinking-busan"
else:
    raise FileNotFoundError("shrinking-busan/data ??? ?? ? ????.")
data_dir = project_dir / "data"

# ?? ?? ?? ?? ??? .env?? V-World ?? ????.
for env_dir in (cwd, *cwd.parents):
    env_path = env_dir / ".env"
    if env_path.exists():
        load_dotenv(env_path, override=False)
        break
vworld_key = os.getenv("V-WORLD_KEY")
if not vworld_key:
    raise ValueError(".env? V-WORLD_KEY? ???? ??? ?????.")

# 2026? 9? ????: ? ???? ???? ???????? ??
living = pd.read_csv(data_dir / "busan_living_population_202609.csv", dtype={"?????": "string"})
resident = pd.read_csv(data_dir / "busan_resident_population_202609.csv", dtype={"??????": "string"})

living["?????"] = living["?????"].str.strip().str.zfill(10)
resident["?????"] = resident["??????"].str.strip().str.zfill(10)
if not living["????"].astype(str).eq("202609").all():
    raise ValueError("???? ??? 2026? 9? ? ??? ???? ????.")
resident = resident.loc[
    resident["????"].astype(str).eq("2026-09-30")
    & resident["???"].eq("?????")
].copy()

pop_cols = ["????_???", "????_???", "????_???"]
for col in pop_cols:
    living[col] = pd.to_numeric(living[col], errors="coerce")
living["???_????"] = living[pop_cols].sum(axis=1, min_count=3)
records_per_dong = living.groupby("?????").size()
if not records_per_dong.eq(24).all():
    raise ValueError("???? ??? ??? 24?? ??? ?????.")

# ???? ??? ???? ??? ?? ??? 24? ?? ?????,
# ? ?? ??? ??? ?? 24? ??? ??? ?????.
living_dong = (
    living.groupby("?????", as_index=False)["???_????"]
    .mean()
    .rename(columns={"???_????": "????_24????"})
)
resident["??????"] = pd.to_numeric(resident["?? ?????"], errors="coerce")
resident_dong = resident[["?????", "??????"]].copy()

boundary = gpd.read_file(data_dir / "busan_hjd_20250101.geojson")
if boundary.crs is None:
    boundary = boundary.set_crs("EPSG:4326")
boundary = boundary.to_crs("EPSG:4326")
boundary["?????"] = boundary["adm_cd2"].astype(str).str.replace(r"\.0$", "", regex=True).str.zfill(10)
boundary["????"] = boundary["adm_nm"]
map_data = boundary.merge(living_dong, on="?????", how="left").merge(
    resident_dong, on="?????", how="left"
)
if len(map_data) != 206 or map_data[["????_24????", "??????"]].isna().any().any():
    raise ValueError("?????????????? ?? ?? ??? ?????.")
map_data["????_???????"] = (
    map_data["????_24????"] / map_data["??????"]
)
labels = ["0.50 ??", "0.50?0.79", "0.80?0.99", "1.00?1.49", "1.50 ??"]
bins = [-np.inf, 0.5, 0.8, 1.0, 1.5, np.inf]
map_data["??_??"] = pd.cut(
    map_data["????_???????"], bins=bins, labels=labels, right=False
).astype(str)
colors = {
    "0.50 ??": "#2166ac",
    "0.50?0.79": "#67a9cf",
    "0.80?0.99": "#d1e5f0",
    "1.00?1.49": "#f4a582",
    "1.50 ??": "#b2182b",
}

center_5179 = map_data.to_crs(epsg=5179).geometry.union_all().centroid
center_4326 = gpd.GeoSeries([center_5179], crs=5179).to_crs(epsg=4326).iloc[0]
m = folium.Map(location=[center_4326.y, center_4326.x], zoom_start=10, tiles=None, control_scale=True)
folium.TileLayer(
    tiles=f"https://api.vworld.kr/req/wmts/1.0.0/{vworld_key}/Base/{{z}}/{{y}}/{{x}}.png",
    attr="? V-World", name="V-World ????", max_zoom=19, control=False,
).add_to(m)

def style_feature(feature):
    return {
        "fillColor": colors.get(feature["properties"].get("??_??"), "#bdbdbd"),
        "color": "#555555", "weight": 0.6, "fillOpacity": 0.78,
    }

folium.GeoJson(
    data=map_data.to_json(),
    name="???? / ?????? ??",
    style_function=style_feature,
    tooltip=folium.GeoJsonTooltip(
        fields=["????", "????_24????", "??????", "????_???????", "??_??"],
        aliases=["???", "????(24?? ??)", "??????(9? ?)", "??", "??"],
        localize=True, sticky=True,
    ),
).add_to(m)
legend_rows = "".join(
    f'<div style="display:flex;align-items:center;gap:7px;margin:3px 0;">'
    f'<span style="width:15px;height:13px;background:{colors[label]};border:1px solid #999"></span>{label}</div>'
    for label in labels
)
legend_html = (
    '<div style="position:fixed;z-index:9999;right:22px;bottom:28px;background:white;'
    'padding:10px 13px;border-radius:6px;box-shadow:0 1px 6px #0003;font-size:12px;">'
    '<b>???? / ??????</b>' + legend_rows + '</div>'
)
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)
preview_path = project_dir / "preview_202609.html"
m.save(str(preview_path))
display(IFrame(src=preview_path.name, width="100%", height=760))
